In [1]:
import torch 
import numpy as np 
import h5py
import os
from pathlib import Path
import importlib
import IPython.display as ipd

import src.spatial_attn_lightning as binaural_lightning 
import yaml
from pytorch_lightning import Trainer

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

torch.set_float32_matmul_precision('medium')

In [2]:

config_path = "config/binaural_attn/word_task_no_co_loc_v05.yml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 2
config['hparas']['batch_size'] = 32
config['audio']['rep_kwargs']['rep_on_gpu'] = True



In [4]:
importlib.reload(binaural_lightning)
module = binaural_lightning.BinauralAttentionModule(config)

num_classes={'num_words': 800}
Model performing word task
center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


In [5]:
module

BinauralAttentionModule(
  (audio_transforms): AudioCompose(
      AudioToTensor()
      BinauralRMSNormalizeForegroundAndBackground()
  )
  (model): OptimizedModule(
    (_orig_mod): CNN2DExtractor(
      (model_dict): ModuleDict(
        (norm_coch_rep): LayerNorm((2, 40, 20000), eps=1e-05, elementwise_affine=True)
        (attn_block_in): SimpleAttentionalGain()
        (conv_block_0): Sequential(
          (0): LayerNorm((2, 40, 20000), eps=1e-05, elementwise_affine=True)
          (1): Conv2dSame(2, 32, kernel_size=(2, 34), stride=(1, 1), bias=False)
          (2): ReLU()
          (3): HannPooling2d()
        )
        (attn0): SimpleAttentionalGain()
        (conv_block_1): Sequential(
          (0): LayerNorm((32, 20, 5000), eps=1e-05, elementwise_affine=True)
          (1): Conv2dSame(32, 64, kernel_size=(2, 14), stride=(1, 1), bias=False)
          (2): ReLU()
          (3): HannPooling2d()
        )
        (attn1): SimpleAttentionalGain()
        (conv_block_2): Sequential(

In [7]:

trainer = Trainer(
    precision="32",
    limit_val_batches=0.0,
    num_nodes=1,
    devices=1, # was gpus=1,
    # detect_anomaly=True,
    # strategy="ddp_notebook",
    accelerator="gpu",
)
trainer.fit(module)

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/lightning_fabric/plugins/environments/slurm.py:191: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /om2/user/imgriff/conda_envs/pytorch_2/lib/python3.1 ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/pytorch_lightning/loops/utilities.py:73: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type                         | Params
------------------------------------------------------------------
0 | audio_transforms | AudioCompose                 | 0     
1 | model            | OptimizedModule      

Using v05 dataset
/om/scratch/Sun/imgriff/datasets/spatial_audio_pipeline/assets/dataset_binaural_attn/v05
cue type: voice_and_location
985 files in train concat dataset
len training set = 123125
Epoch 0:   0%|          | 8/123125 [00:49<212:57:55,  0.16it/s, v_num=3.42e+7, train_loss=6.680] 

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/pytorch_lightning/trainer/call.py:54: Detected KeyboardInterrupt, attempting graceful shutdown...
